In [1]:
import torch
import itertools
import os
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

In [2]:
# elbo_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_06-30_non_amortized_vae_output/")
elbo_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_07-11_non_amortized_vae_larger_init_range_output/")
favi_result_dir = Path("/data/scratch/pduan/new_gcvi_output/gcvi_06-30_set_transformer_favi_output/")

In [3]:
elbo_not_converge_cases = []
favi_not_converge_cases = []
n_cases = 0
result_files = os.listdir(elbo_result_dir)
for i, item in enumerate(result_files):
    elbo_full_path = elbo_result_dir / item
    assert elbo_full_path.exists()
    elbo_results = torch.load(elbo_full_path, map_location="cpu")
    task_name = elbo_results[0]["task"]
    favi_full_path = favi_result_dir / item
    if not favi_full_path.exists():
        print("=" * 50)
        print(f"[{i + 1}] task name: {task_name}")
        print("can't find the favi file")
        continue
    else:
        n_cases += 1
    favi_results = torch.load(favi_full_path, map_location="cpu")
    elbo_not_converge = 0
    favi_not_converge = 0
    for r in elbo_results:
        if r["elbo_error"] is not None:
            elbo_not_converge += 1
    for r in favi_results:
        if r["favi_error"] is not None:
            favi_not_converge += 1
    if elbo_not_converge > 0:
        elbo_not_converge_cases.append(task_name)
    if favi_not_converge > 0:
        favi_not_converge_cases.append(task_name)
    elbo_results_num = len(elbo_results)
    favi_results_num = len(favi_results)
    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")
    print(f"elbo can't converge: {elbo_not_converge} / {elbo_results_num}")
    print(f"favi can't converge: {favi_not_converge} / {favi_results_num}")
print("=" * 100)
print("Summary:")
print(f"elbo not converge cases: {len(elbo_not_converge_cases)} / {n_cases} ({len(elbo_not_converge_cases) / n_cases:.4f})")
print(f"elbo not converge cases: {elbo_not_converge_cases}")
print(f"favi not converge cases: {len(favi_not_converge_cases)} / {n_cases} ({len(favi_not_converge_cases) / n_cases:.4f})")
print(f"favi not converge cases: {favi_not_converge_cases}")

[1] task name: arm_kidiq_interaction_c
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[2] task name: arm_electric_multi_preds
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[3] task name: arm_kidiq_interaction
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[4] task name: arm_electric_1b_chr
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[5] task name: arm_radon_no_pool
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[6] task name: arm_radon_group_chr
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[7] task name: arm_sesame_one_pred_a
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[8] task name: arm_latent_glm
elbo can't converge: 96 / 96
favi can't converge: 0 / 96
[9] task name: arm_wells
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[10] task name: arm_anova_randon_nopred
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[11] task name: arm_congress
elbo can't converge: 0 / 96
favi can't converge: 0 / 96
[12

In [4]:
def get_est_mu_sigma2(results, tag):
    est_mu = []
    for r in results:
        est_mu.append(r[f"{tag}_test_dict_list"]["est_mu"])  # (num_obs, k)
    est_mu = torch.stack(est_mu, dim=0)  # (r, num_obs, k)
    est_sigma2 = []
    for r in results:
        est_sigma2.append(r[f"{tag}_test_dict_list"]["est_sigma2"])  # (num_obs, k)
    est_sigma2 = torch.stack(est_sigma2, dim=0)  # (r, num_obs, k)
    return est_mu, est_sigma2

In [5]:
def kl_div_two_normal(p_mu, p_sigma2, q_mu, q_sigma2):
    return torch.log(q_sigma2.sqrt()) - torch.log(p_sigma2.sqrt()) + (p_sigma2 + (p_mu - q_mu) ** 2) / (2 * q_sigma2) - 0.5

In [6]:
def get_kl_for_repeats(est_mu_sigma2: torch.Tensor):
    # est_mu_sigma2: (r, num_obs, k, 2)
    assert est_mu_sigma2.shape[-1] == 2
    combs = torch.tensor(list(itertools.combinations(range(est_mu_sigma2.shape[0]), 2)))  # (c, 2)
    perms = torch.cat([combs, combs.flip(dims=[-1])], dim=0)  # (2c, 2)
    perm_est_mu_sigma2 = est_mu_sigma2[perms]  # (2c, 2, num_obs, k, 2)
    return kl_div_two_normal(p_mu=perm_est_mu_sigma2[:, 0, :, :, 0],
                            p_sigma2=perm_est_mu_sigma2[:, 0, :, :, 1],
                            q_mu=perm_est_mu_sigma2[:, 1, :, :, 0],
                            q_sigma2=perm_est_mu_sigma2[:, 1, :, :, 1])  # (2c, num_obs, k)

In [7]:
def wasserstein_distance_to_unif(u: torch.Tensor):
    assert u.ndim == 3  # (r, k, s)
    unif_samples = torch.linspace(0.0, 1.0, u.shape[-1]).view(1, 1, -1)
    sorted_u = torch.sort(u, dim=-1, descending=False)[0]  # (r, k, s)
    return torch.abs(sorted_u - unif_samples).mean(dim=-1)  # (r, k)

In [8]:
def extract_vsbc(results, tag):
    vsbc_list = []
    for r in results:
        vsbc_list.append(r[f"{tag}_vsbc"])  # (k, s)
    return torch.stack(vsbc_list, dim=0)  # (r, k, s)

In [9]:
result_files = os.listdir(elbo_result_dir)
elbo_avg_kl_r = []
elbo_std_mu = []
elbo_std_sigma2 = []
elbo_vsbc_to_unif_dist = []
favi_avg_kl_r = []
favi_std_mu = []
favi_std_sigma2 = []
favi_vsbc_to_unif_dist = []
favi_elbo_kl_ratio = []
favi_elbo_std_mu_ratio = []
favi_elbo_std_sigma2_ratio = []
favi_elbo_vsbc_d_ratio = []
task_name_list = []
n_cases = 0
for i, item in enumerate(result_files):
    elbo_full_path = elbo_result_dir / item
    assert elbo_full_path.exists()
    elbo_results = torch.load(elbo_full_path, map_location="cpu")
    task_name = elbo_results[0]["task"]
    favi_full_path = favi_result_dir / item
    if not favi_full_path.exists():
        print("=" * 50)
        print(f"[{i + 1}] task name: {task_name}")
        print("can't find the favi file")
        continue
    else:
        n_cases += 1
    favi_results = torch.load(favi_full_path, map_location="cpu")
    task_name_list.append(task_name)
    find_elbo_error = any([r["elbo_error"] is not None for r in elbo_results])
    find_favi_error = any([r["favi_error"] is not None for r in favi_results])
    print("=" * 50)
    print(f"[{i + 1}] task name: {task_name}")
    print(f"elbo error: {find_favi_error}")
    print(f"favi error: {find_elbo_error}")

    if not find_elbo_error:
        elbo_est_mu, elbo_est_sigma2 = get_est_mu_sigma2(elbo_results, tag="elbo")
        assert (elbo_est_sigma2 > 0).all()
        print(f"std(elbo_est_mu): {elbo_est_mu.std(dim=0).mean().item():.3e}")
        elbo_std_mu.append(elbo_est_mu.std(dim=0).mean().item())
        print(f"std(elbo_est_sigma2): {elbo_est_sigma2.std(dim=0).mean().item():.3e}")
        elbo_std_sigma2.append(elbo_est_sigma2.std(dim=0).mean().item())
        elbo_kl_r = get_kl_for_repeats(torch.stack([elbo_est_mu, elbo_est_sigma2], dim=-1))
        print(f"elbo avg kl for repeats: {elbo_kl_r.mean().item():.3e}")
        elbo_avg_kl_r.append(elbo_kl_r.mean().item())
        elbo_vsbc = extract_vsbc(elbo_results, tag="elbo")  # (r, k, s)
        elbo_vsbc_d = wasserstein_distance_to_unif(elbo_vsbc)  # (r, k)
        print(f"elbo vsbc wasserstein dist to unif: {elbo_vsbc_d.mean().item():.3e}")
        elbo_vsbc_to_unif_dist.append(elbo_vsbc_d.mean().item())
    else:
        print("skip elbo, due to elbo error")
        elbo_std_mu.append(torch.nan)
        elbo_std_sigma2.append(torch.nan)
        elbo_avg_kl_r.append(torch.nan)
        elbo_vsbc_to_unif_dist.append(torch.nan)
    
    if not find_favi_error:
        favi_est_mu, favi_est_sigma2 = get_est_mu_sigma2(favi_results, tag="favi")
        assert (favi_est_sigma2 > 0).all()
        print(f"std(favi_est_mu): {favi_est_mu.std(dim=0).mean().item():.3e}")
        favi_std_mu.append(favi_est_mu.std(dim=0).mean().item())
        print(f"std(favi_est_sigma2): {favi_est_sigma2.std(dim=0).mean().item():.3e}")
        favi_std_sigma2.append(favi_est_sigma2.std(dim=0).mean().item())
        favi_kl_r = get_kl_for_repeats(torch.stack([favi_est_mu, favi_est_sigma2], dim=-1))
        print(f"favi avg kl for repeats: {favi_kl_r.mean().item():.3e}")
        favi_avg_kl_r.append(favi_kl_r.mean().item())
        favi_vsbc = extract_vsbc(favi_results, tag="favi")  # (r, k, s)
        favi_vsbc_d = wasserstein_distance_to_unif(favi_vsbc)  # (r, k)
        print(f"favi vsbc wasserstein dist to unif: {favi_vsbc_d.mean().item():.3e}")
        favi_vsbc_to_unif_dist.append(favi_vsbc_d.mean().item())
    else:
        print("skip favi, due to favi error")
        favi_std_mu.append(torch.nan)
        favi_std_sigma2.append(torch.nan)
        favi_avg_kl_r.append(torch.nan)
        favi_vsbc_to_unif_dist.append(torch.nan)

    if not find_elbo_error and not find_favi_error:
        if elbo_est_mu.std(dim=0).mean().item() > 0:
            favi_elbo_std_mu_ratio.append(favi_est_mu.std(dim=0).mean().item() / elbo_est_mu.std(dim=0).mean().item())
        if elbo_est_sigma2.std(dim=0).mean().item() > 0:
            favi_elbo_std_sigma2_ratio.append(favi_est_sigma2.std(dim=0).mean().item() / elbo_est_sigma2.std(dim=0).mean().item())
        favi_elbo_kl_ratio.append(favi_kl_r.mean().item() / elbo_kl_r.mean().item())
        favi_elbo_vsbc_d_ratio.append(favi_vsbc_d.mean().item() / elbo_vsbc_d.mean().item())

print()
print("+" * 100)
print("Summary:")
elbo_avg_kl_r = torch.tensor(elbo_avg_kl_r)
assert not torch.isinf(elbo_avg_kl_r).any()
elbo_std_mu = torch.tensor(elbo_std_mu)
assert not torch.isinf(elbo_std_mu).any()
elbo_std_sigma2 = torch.tensor(elbo_std_sigma2)
assert not torch.isinf(elbo_std_sigma2).any()
elbo_vsbc_to_unif_dist = torch.tensor(elbo_vsbc_to_unif_dist)
assert not torch.isinf(elbo_vsbc_to_unif_dist).any()

favi_avg_kl_r = torch.tensor(favi_avg_kl_r)
assert not torch.isinf(favi_avg_kl_r).any()
favi_std_mu = torch.tensor(favi_std_mu)
assert not torch.isinf(favi_std_mu).any()
favi_std_sigma2 = torch.tensor(favi_std_sigma2)
assert not torch.isinf(favi_std_sigma2).any()
favi_vsbc_to_unif_dist = torch.tensor(favi_vsbc_to_unif_dist)
assert not torch.isinf(favi_vsbc_to_unif_dist).any()

check_less_than = lambda a, b: ((a < b).sum() - torch.isnan(a).sum() + torch.isnan(b).sum()).item()
def print_less_than_info(favi_d, elbo_d, tag):
    print(f"favi {tag} < elbo {tag}: "\
            f"{check_less_than(favi_d, elbo_d)}/{n_cases} "\
            f"({check_less_than(favi_d, elbo_d) / n_cases:.3f})")
    not_less_than = ~(favi_d < elbo_d)
    not_less_than_cases = [t for i, t in enumerate(task_name_list) if not_less_than[i]]
    print(f"favi {tag} >= elbo {tag} cases: {not_less_than_cases}")
    favi_nan = torch.isnan(favi_d)
    favi_nan_cases = [t for i, t in enumerate(task_name_list) if favi_nan[i]]
    print(f"favi {tag} nan cases: {favi_nan_cases}")
    elbo_nan = torch.isnan(elbo_d)
    elbo_nan_cases = [t for i, t in enumerate(task_name_list) if elbo_nan[i]]
    print(f"elbo {tag} nan cases: {elbo_nan_cases}")
    return [result_files[i] 
            for i in range(len(task_name_list)) 
            if not_less_than[i] and (not favi_nan[i]) and (not elbo_nan[i])]
std_mu_not_less_than_cases = print_less_than_info(favi_std_mu, elbo_std_mu, tag="std(mu)")
std_sigma2_not_less_than_cases = print_less_than_info(favi_std_sigma2, elbo_std_sigma2, tag="std(sigma2)")
kl_not_less_than_cases = print_less_than_info(favi_avg_kl_r, elbo_avg_kl_r, tag="kl")
vsbc_d_not_less_than_cases = print_less_than_info(favi_vsbc_to_unif_dist, elbo_vsbc_to_unif_dist, tag="vsbc to unif dist")

favi_elbo_std_mu_ratio = torch.tensor(favi_elbo_std_mu_ratio)
favi_elbo_std_sigma2_ratio = torch.tensor(favi_elbo_std_sigma2_ratio)
favi_elbo_kl_ratio = torch.tensor(favi_elbo_kl_ratio)
favi_elbo_vsbc_d_ratio = torch.tensor(favi_elbo_vsbc_d_ratio)
# print(f"mean of std(mu) ratio: {favi_elbo_std_mu_ratio.mean().item():.3e}")
print(f"median of std(mu) ratio: {favi_elbo_std_mu_ratio.median().item():.3e}")
# print(f"mean of std(sigma2) ratio: {favi_elbo_std_sigma2_ratio.mean().item():.3e}")
print(f"median of std(sigma2) ratio: {favi_elbo_std_sigma2_ratio.median().item():.3e}")
# print(f"mean of kl ratio: {favi_elbo_kl_ratio.mean().item():.3e}")
print(f"median of kl ratio: {favi_elbo_kl_ratio.median().item():.3e}")
# print(f"mean of vsbc d ratio: {favi_elbo_vsbc_d_ratio.mean().item():.3e}")
print(f"median of vsbc d ratio: {favi_elbo_vsbc_d_ratio.median().item():.3e}")
print("+" * 100)

[1] task name: arm_kidiq_interaction_c
elbo error: False
favi error: False
std(elbo_est_mu): 5.302e+00
std(elbo_est_sigma2): 1.681e+02
elbo avg kl for repeats: 1.579e+01
elbo vsbc wasserstein dist to unif: 4.112e-02
std(favi_est_mu): 2.541e+00
std(favi_est_sigma2): 1.254e-01
favi avg kl for repeats: 1.680e+00
favi vsbc wasserstein dist to unif: 8.859e-02
[2] task name: arm_electric_multi_preds
elbo error: False
favi error: False
std(elbo_est_mu): 5.442e+00
std(elbo_est_sigma2): 5.855e+02
elbo avg kl for repeats: 3.918e+01
elbo vsbc wasserstein dist to unif: 1.003e-01
std(favi_est_mu): 7.630e-02
std(favi_est_sigma2): 2.078e-02
favi avg kl for repeats: 3.682e-01
favi vsbc wasserstein dist to unif: 5.907e-02
[3] task name: arm_kidiq_interaction
elbo error: False
favi error: False
std(elbo_est_mu): 5.492e+00
std(elbo_est_sigma2): 1.280e+03
elbo avg kl for repeats: 7.296e+01
elbo vsbc wasserstein dist to unif: 9.058e-02
std(favi_est_mu): 5.757e-01
std(favi_est_sigma2): 2.368e-02
favi avg kl

## Test Two Moons

In [10]:
elbo_two_moons = torch.load(elbo_result_dir / "pyro_t_two_moons_mn_96.pt", map_location="cpu")
favi_two_moons = torch.load(favi_result_dir / "pyro_t_two_moons_mn_96.pt", map_location="cpu")

In [11]:
len(elbo_two_moons), len(favi_two_moons)

(96, 96)

In [12]:
elbo_two_moons[0]["elbo_test_dict_list"]["obs"].shape, favi_two_moons[0]["favi_test_dict_list"]["obs"].shape

(torch.Size([1000, 2]), torch.Size([1000, 2, 1]))

In [13]:
torch.allclose(elbo_two_moons[0]["elbo_test_dict_list"]["obs"], 
               favi_two_moons[0]["favi_test_dict_list"]["obs"].squeeze(-1))

True

In [14]:
torch.allclose(elbo_two_moons[0]["elbo_test_dict_list"]["obs"], 
               elbo_two_moons[95]["elbo_test_dict_list"]["obs"])

True

In [15]:
torch.allclose(favi_two_moons[0]["favi_test_dict_list"]["obs"], 
               favi_two_moons[95]["favi_test_dict_list"]["obs"])

True

In [16]:
elbo_two_moons[0]["elbo_test_dict_list"]["est_mu"].shape, favi_two_moons[0]["favi_test_dict_list"]["est_mu"].shape

(torch.Size([1000, 2]), torch.Size([1000, 2]))

In [17]:
elbo_two_moons[0]["elbo_test_dict_list"]["est_mu"][:5]

tensor([[-3.7675,  8.1833],
        [ 8.3566,  3.6335],
        [-5.3415,  0.0688],
        [-8.0547, -7.6551],
        [ 5.3986, -4.9381]])

In [18]:
elbo_two_moons[1]["elbo_test_dict_list"]["est_mu"][:5]

tensor([[-2.4281,  1.2646],
        [ 5.5441, -0.5770],
        [-0.2653,  0.0451],
        [ 0.7225,  2.9527],
        [-7.7766, -1.6493]])

In [19]:
favi_two_moons[0]["favi_test_dict_list"]["est_mu"][:5]

tensor([[-0.2344,  0.2364],
        [ 0.0017, -0.0025],
        [-0.1062,  0.1050],
        [-0.0904,  0.0896],
        [-0.0204,  0.0199]])

In [20]:
favi_two_moons[1]["favi_test_dict_list"]["est_mu"][:5]

tensor([[-0.2431,  0.2439],
        [ 0.0047, -0.0069],
        [-0.1062,  0.1055],
        [-0.1001,  0.0993],
        [-0.0221,  0.0183]])

In [21]:
elbo_two_moons[0]["elbo_test_dict_list"]["est_sigma2"][:5]

tensor([[25.8187, 78.5880],
        [75.2667, 17.7886],
        [39.5510,  0.3647],
        [40.6052, 95.4513],
        [32.5431, 36.1376]])

In [22]:
elbo_two_moons[1]["elbo_test_dict_list"]["est_sigma2"][:5]

tensor([[16.8854,  4.0508],
        [36.7091,  1.4852],
        [ 0.2435,  0.1415],
        [ 8.2582, 12.9713],
        [62.9347, 13.7392]])

In [23]:
favi_two_moons[0]["favi_test_dict_list"]["est_sigma2"][:5]

tensor([[0.0055, 0.0055],
        [0.1171, 0.1186],
        [0.0222, 0.0221],
        [0.0863, 0.0831],
        [0.0441, 0.0441]])

In [24]:
favi_two_moons[1]["favi_test_dict_list"]["est_sigma2"][:5]

tensor([[0.0055, 0.0056],
        [0.1212, 0.1172],
        [0.0215, 0.0225],
        [0.0813, 0.0786],
        [0.0447, 0.0440]])